# Support-ticket triage with TypeSafe Jev
Jev is a System-1 model: it returns typed answers (choice / score / noul) with probabilities, not text. Get a key from typesafe.ai, then add it in Colab **Secrets** (🔑 sidebar) as `TYPESAFE_API_KEY`.

In [ ]:
!pip -q install typesafe-sdk pandas

In [ ]:
import os
from google.colab import userdata
os.environ["TYPESAFE_API_KEY"] = userdata.get("TYPESAFE_API_KEY")

In [ ]:
from typesafe_sdk import Choice, Noul, Score, TypeSafeClient
client = TypeSafeClient()

QUESTIONS = {
    "department": Choice(instructions="Which team handles this ticket?",
        criteria={"billing": "Payments, refunds, invoices", "technical": "Bugs, errors, outages", "account": "Login, profile, access"}),
    "urgent": Noul(instructions="Is this urgent?"),
    "frustration": Score(instructions="Customer frustration level",
        criteria=["Calm", "Frustrated", "Very angry"]),
}

def triage(ticket):
    a = client.system_one(state={"ticket": ticket}, questions=QUESTIONS).answers
    return {"department": a["department"].choice, "dept_conf": a["department"].confidence,
            "urgent_p": a["urgent"].noul, "frustration": a["frustration"].score}

In [ ]:
import pandas as pd
tickets = [
    "I was charged twice for my subscription this month!!",
    "The app crashes every time I open settings.",
    "Can't log in, password reset email never arrives.",
    "Site is completely down and we're losing sales. FIX NOW.",
    "Thanks, just checking when my invoice is issued.",
]
df = pd.DataFrame([{"ticket": t, **triage(t)} for t in tickets])
df

In [ ]:
# Jev is fast: run in a loop / route by confidence
for _, r in df.iterrows():
    route = r.department if r.dept_conf > 0.6 else "human-review"
    print(f"{route:13} urgent={r.urgent_p:.2f}  {r.ticket[:50]}")